
# CARE-MoE V3 — Experiment 1: Do Existing Similarity Metrics Predict Merge-Induced Capability Loss?

**Research question (the only thing this notebook answers):**
> Can existing expert-similarity metrics predict the true capability loss caused by
> merging experts in a pretrained Mixture-of-Experts model?

**Explicit non-goals for this notebook (do not violate these while editing):**
- Do **not** design a new compression algorithm.
- Do **not** implement CARE.
- Do **not** invent a new proxy metric.

Everything here is either (a) a well-known, pre-existing similarity metric, or
(b) the *oracle* ground truth obtained by actually merging two experts and
measuring the real output degradation. The only thing being tested is whether
(a) correlates with (b).

**Requires an NVIDIA GPU (CUDA).** The oracle sweep runs two full forward
passes over the calibration set (baseline + merged) *per expert pair*, so it
is intentionally compute-heavy — see the runtime note before the oracle cell.


## 1. Configuration

In [ ]:

import os, math, copy, random, itertools

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# CONFIG — the only knobs you should need to touch
# ============================================================
MODEL_ID   = "Qwen/Qwen1.5-MoE-A2.7B"
SEED       = 42
NUM_SEQUENCES = 256          # calibration sequences (spec requirement)
MAX_LENGTH    = 512          # tokens per sequence (spec requirement)

LAYER_INDEX = 0              # <-- which MoE layer to evaluate (0-indexed)

# Oracle sweep is O(num_expert_pairs) full forward passes and is expensive.
# Qwen1.5-MoE-A2.7B has 60 routed experts per layer -> C(60,2) = 1770 pairs.
# Set MAX_PAIRS to a small int (e.g. 20) for a smoke test before committing
# to the full sweep. Set to None to run every pair (the actual experiment).
MAX_PAIRS = None

# Batch size used for calibration forward passes (memory/speed tradeoff only —
# does not change any metric definition).
CALIB_BATCH_SIZE = 4

# Number of calibration tokens used for the (cheap, one-forward-per-expert)
# output/activation similarity metrics. These are computed ONCE per expert
# (not per pair) and then compared pairwise, so this can be generous without
# blowing up runtime. Reduce this first if you hit CPU/GPU memory limits.
EVAL_TOKENS_FOR_EXPERT_METRICS = 4096

DEVICE = "cuda"
DTYPE  = torch.float16

OUTPUT_DIR  = "./care_moe_v3_outputs"
SCATTER_DIR = os.path.join(OUTPUT_DIR, "scatterplots")
os.makedirs(SCATTER_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."
print("CUDA device:", torch.cuda.get_device_name(0))


## 2. Model — load Qwen1.5-MoE-A2.7B in FP16 on CUDA

In [ ]:

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in FP16 on CUDA (this is a ~28GB download, cached after first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map={"": 0},   # force the WHOLE model onto one GPU — no CPU/disk offload.
                          # We swap individual expert sub-modules in place later; that
                          # is only safe/simple if the whole model lives on one device.
)
model.eval()
print("Model loaded.")


## 3. Detect all MoE layers and print architecture summary

In [ ]:

def find_moe_layers(model):
    '''
    Generic detector (not hard-coded to Qwen2Moe internals): returns every
    sub-module in the model that exposes a ModuleList called `experts`.
    This is what lets LAYER_INDEX below refer to "the Nth MoE layer" without
    hand-coding transformer block indices.
    '''
    found = []
    for name, module in model.named_modules():
        if hasattr(module, "experts") and isinstance(module.experts, torch.nn.ModuleList):
            found.append((name, module))
    return found

moe_layers = find_moe_layers(model)
cfg = model.config

print("=" * 70)
print("MoE ARCHITECTURE SUMMARY")
print("=" * 70)
print(f"Number of MoE layers       : {len(moe_layers)}")
print(f"Experts per layer          : {len(moe_layers[0][1].experts)}")
print(f"Hidden dimension           : {cfg.hidden_size}")
print(f"Expert (FFN) dimension     : {getattr(cfg, 'moe_intermediate_size', getattr(cfg, 'intermediate_size', 'n/a'))}")
print(f"Routing type               : top-k softmax gating (learned linear router per layer)")
print(f"Top-k configuration        : {getattr(cfg, 'num_experts_per_tok', 'n/a')}")
print(f"Top-k prob renormalization : {getattr(cfg, 'norm_topk_prob', 'n/a')}")
print(f"Shared (always-on) expert  : {'yes' if hasattr(moe_layers[0][1], 'shared_expert') else 'no'} "
      f"(excluded from this analysis — we only study ROUTED experts)")
print("=" * 70)


## 4. Dataset — WikiText-2 calibration set (256 sequences x 512 tokens, seed=42)

In [ ]:

print("Loading WikiText-2...")
raw = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# Keep non-empty lines only (WikiText has many blank/section-header rows)
texts = [t for t in raw["text"] if len(t.strip()) > 0]

rng = random.Random(SEED)
rng.shuffle(texts)

tokenized_sequences = []
for t in texts:
    if len(tokenized_sequences) >= NUM_SEQUENCES:
        break
    enc = tokenizer(
        t,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    # Skip sequences that are almost entirely padding — they carry ~no signal
    # for calibration and would just dilute every metric with padding tokens.
    if enc["attention_mask"].sum().item() < 8:
        continue
    tokenized_sequences.append(enc)

assert len(tokenized_sequences) == NUM_SEQUENCES, (
    f"Only found {len(tokenized_sequences)} usable sequences out of the requested "
    f"{NUM_SEQUENCES}. WikiText-2 train split should have far more than enough — "
    f"if you see this, something upstream (tokenizer/config) is off."
)

calib_input_ids = torch.cat([s["input_ids"] for s in tokenized_sequences], dim=0)
calib_attn_mask = torch.cat([s["attention_mask"] for s in tokenized_sequences], dim=0)

print(f"Calibration set: {calib_input_ids.shape[0]} sequences x {calib_input_ids.shape[1]} tokens "
      f"= {int(calib_attn_mask.sum().item())} real (non-pad) tokens")


## 5. Layer selection (single configurable layer)

In [ ]:

layer_name, moe_block = moe_layers[LAYER_INDEX]
experts = moe_block.experts
num_experts = len(experts)
print(f"Selected layer index {LAYER_INDEX} -> module '{layer_name}' ({num_experts} routed experts)")



## 6. Baseline merge operator — simple parameter averaging (modular)

This is deliberately the *simplest possible* merge operator, per the experiment
spec. It is wrapped behind an interface so a different operator can be dropped
in later (e.g. for a follow-up experiment) without touching any metric or
oracle code below.


In [ ]:

class MergeOperator:
    '''Abstract interface — swap the concrete class below to test a different
    merge strategy later. Nothing downstream depends on the implementation.'''
    def merge(self, expert_a: torch.nn.Module, expert_b: torch.nn.Module) -> torch.nn.Module:
        raise NotImplementedError


class ParameterAveragingMerge(MergeOperator):
    '''
    BASELINE operator (spec requirement): elementwise average of every
    parameter tensor between two experts. Nothing more sophisticated —
    this experiment exists to test EXISTING similarity metrics against this
    baseline, not to build a better merge function.
    '''
    def merge(self, expert_a, expert_b):
        merged = copy.deepcopy(expert_a)
        with torch.no_grad():
            for (name_m, p_m), (name_a, p_a), (name_b, p_b) in zip(
                merged.named_parameters(),
                expert_a.named_parameters(),
                expert_b.named_parameters(),
            ):
                assert name_m == name_a == name_b, "Parameter name mismatch between experts"
                p_m.copy_(0.5 * p_a.float() + 0.5 * p_b.float())
        return merged


merge_operator = ParameterAveragingMerge()  # <-- swap this line to test a different operator later



## 7. Existing metrics, part A — weight-space metrics (Euclidean distance, cosine similarity)

Computed directly from parameter tensors. No calibration data needed.


In [ ]:

def flatten_expert_weights(expert):
    return torch.cat([p.detach().flatten().float() for p in expert.parameters()])

flat_weights = [flatten_expert_weights(e) for e in experts]

def weight_distance(i, j):
    return torch.norm(flat_weights[i] - flat_weights[j]).item()

def weight_cosine(i, j):
    return F.cosine_similarity(flat_weights[i].unsqueeze(0), flat_weights[j].unsqueeze(0)).item()



## 8. Existing metrics, part B — capture router logits and layer-input activations

We run the *full, unmodified* model over the calibration set exactly once and
hook the selected MoE layer to capture:
- the hidden states entering that layer (per real, non-padding token), and
- the router's logits for those same tokens.

Everything downstream (usage frequency, routing similarity, and the
per-expert forward passes for output/activation similarity) reuses these
captured tensors — we only pay for one full-model forward pass over the
calibration set for all of this.


In [ ]:

@torch.no_grad()
def collect_calibration_activations(model, moe_block, input_ids, attention_mask, batch_size):
    hidden_list, router_list, spans = [], [], []

    def pre_hook(module, inputs):
        hidden_list.append(inputs[0].detach())

    def gate_hook(module, inputs, output):
        router_list.append(output.detach())

    h1 = moe_block.register_forward_pre_hook(pre_hook)
    h2 = moe_block.gate.register_forward_hook(gate_hook)

    n = input_ids.shape[0]
    for start in tqdm(range(0, n, batch_size), desc="Capturing layer activations"):
        end = min(start + batch_size, n)
        ids = input_ids[start:end].to(DEVICE)
        mask = attention_mask[start:end].to(DEVICE)
        model(input_ids=ids, attention_mask=mask)
        spans.append((start, end))

    h1.remove()
    h2.remove()

    hidden_all, router_all = [], []
    for hs, rl, (start, end) in zip(hidden_list, router_list, spans):
        mask = attention_mask[start:end]
        b, seq, hid = hs.shape
        hs_flat = hs.reshape(b * seq, hid)
        mask_flat = mask.reshape(b * seq).bool()
        hidden_all.append(hs_flat[mask_flat].cpu())
        router_all.append(rl.reshape(b * seq, -1)[mask_flat].cpu())

    return torch.cat(hidden_all, dim=0), torch.cat(router_all, dim=0)


calib_hidden, calib_router_logits = collect_calibration_activations(
    model, moe_block, calib_input_ids, calib_attn_mask, batch_size=CALIB_BATCH_SIZE
)
print(f"Captured {calib_hidden.shape[0]} valid token activations at layer {LAYER_INDEX} "
      f"(hidden dim {calib_hidden.shape[1]}, router logits over {calib_router_logits.shape[1]} experts)")



## 9. Existing metrics, part C — routing similarity and expert usage frequency

Derived purely from the captured router logits — no additional forward passes.


In [ ]:

router_probs = F.softmax(calib_router_logits.float(), dim=-1)  # (T, num_experts)
topk = getattr(cfg, "num_experts_per_tok", 4)
topk_idx = torch.topk(router_probs, k=topk, dim=-1).indices     # (T, k)

usage_counts = torch.zeros(num_experts)
for e in range(num_experts):
    usage_counts[e] = (topk_idx == e).any(dim=-1).sum().item()
usage_frequency = (usage_counts / calib_router_logits.shape[0]).numpy()  # fraction of tokens routed to each expert

def routing_similarity(i, j):
    '''Pearson correlation between the router's raw (softmax) score for expert i
    vs expert j across every calibration token — do the two experts tend to be
    preferred for the same tokens?'''
    r, _ = pearsonr(router_probs[:, i].numpy(), router_probs[:, j].numpy())
    return r

print("Usage frequency range across experts:",
      f"min={usage_frequency.min():.4f}, max={usage_frequency.max():.4f}, mean={usage_frequency.mean():.4f}")



## 10. Existing metrics, part D — output similarity and activation similarity

For each expert (once — not once per pair), run it directly on a shared
subsample of calibration hidden states, bypassing the router entirely. This
gives every expert's output/activation on the *identical* input tokens, so
pairwise similarity is a fair comparison of expert *function*, not of which
tokens happened to be routed where.

- **Output similarity**: cosine similarity between two experts' final outputs,
  averaged over tokens.
- **Activation similarity**: cosine similarity between the intermediate
  gated activation (input to `down_proj`, i.e. `act_fn(gate_proj(x)) * up_proj(x)`),
  averaged over tokens.


In [ ]:

torch.manual_seed(SEED)  # local determinism for the subsample choice
n_avail = calib_hidden.shape[0]
n_eval = min(EVAL_TOKENS_FOR_EXPERT_METRICS, n_avail)
subsample_idx = torch.randperm(n_avail)[:n_eval]
eval_hidden = calib_hidden[subsample_idx].to(DEVICE, dtype=DTYPE)

expert_outputs = {}      # idx -> (n_eval, hidden_dim) fp16 CPU tensor
expert_activations = {}  # idx -> (n_eval, ffn_dim)    fp16 CPU tensor

with torch.no_grad():
    for idx, expert in enumerate(tqdm(experts, desc="Per-expert forward (output/activation)")):
        captured = {}

        def act_pre_hook(module, inputs, _captured=captured):
            _captured["act"] = inputs[0].detach()

        h = expert.down_proj.register_forward_pre_hook(act_pre_hook)
        out = expert(eval_hidden)
        h.remove()

        expert_outputs[idx] = out.detach().half().cpu()
        expert_activations[idx] = captured["act"].detach().half().cpu()

def output_similarity(i, j):
    a = expert_outputs[i].float()
    b = expert_outputs[j].float()
    return F.cosine_similarity(a, b, dim=-1).mean().item()

def activation_similarity(i, j):
    a = expert_activations[i].float()
    b = expert_activations[j].float()
    return F.cosine_similarity(a, b, dim=-1).mean().item()



## 11. Oracle — ground-truth capability loss from actually merging each pair

For every expert pair (i, j):
1. Save both experts' original weights (for restoration).
2. Build the merged expert once via `merge_operator`.
3. Stream through the calibration set batch by batch. For **each batch**,
   run the model twice: once with experts i/j untouched (baseline) and once
   with both slots pointed at the merged expert. Compute KL divergence,
   cross-entropy, and top-1 agreement for that batch, then discard the
   logits immediately.
4. Restore the original experts before moving to the next pair.

**Why recompute the baseline every time instead of caching it:** caching the
full (sequences x tokens x vocab) baseline distribution for this calibration
set would need on the order of 40GB — not a reasonable footprint for a
single-GPU script. Recomputing it per pair is exact (not an approximation),
it just trades compute time for memory safety. This is also why the oracle
sweep is the slow part of this notebook — expect it to dominate total runtime.


In [ ]:

def install_experts(moe_block, idx_list, modules):
    for idx, mod in zip(idx_list, modules):
        moe_block.experts[idx] = mod


@torch.no_grad()
def run_oracle_pair(model, moe_block, i, j, merge_operator, input_ids, attention_mask, batch_size):
    orig_i = moe_block.experts[i]
    orig_j = moe_block.experts[j]

    merged_expert = merge_operator.merge(orig_i, orig_j).to(DEVICE, dtype=DTYPE)

    n = input_ids.shape[0]
    total_tokens = 0
    kl_sum = 0.0
    ce_orig_sum = 0.0
    ce_merged_sum = 0.0
    top1_agree_sum = 0

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        ids = input_ids[start:end].to(DEVICE)
        mask = attention_mask[start:end].to(DEVICE)

        install_experts(moe_block, [i, j], [orig_i, orig_j])
        logits_orig = model(input_ids=ids, attention_mask=mask).logits

        install_experts(moe_block, [i, j], [merged_expert, merged_expert])
        logits_merged = model(input_ids=ids, attention_mask=mask).logits

        shift_logits_orig = logits_orig[:, :-1, :]
        shift_logits_merged = logits_merged[:, :-1, :]
        shift_labels = ids[:, 1:]
        shift_mask = mask[:, 1:].bool()

        logp_orig = F.log_softmax(shift_logits_orig.float(), dim=-1)
        logp_merged = F.log_softmax(shift_logits_merged.float(), dim=-1)
        p_orig = logp_orig.exp()

        kl_tok = (p_orig * (logp_orig - logp_merged)).sum(dim=-1)
        kl_sum += kl_tok[shift_mask].sum().item()

        ce_orig_tok = F.nll_loss(
            logp_orig.reshape(-1, logp_orig.size(-1)), shift_labels.reshape(-1), reduction="none"
        ).reshape(shift_labels.shape)
        ce_merged_tok = F.nll_loss(
            logp_merged.reshape(-1, logp_merged.size(-1)), shift_labels.reshape(-1), reduction="none"
        ).reshape(shift_labels.shape)
        ce_orig_sum += ce_orig_tok[shift_mask].sum().item()
        ce_merged_sum += ce_merged_tok[shift_mask].sum().item()

        top1_orig = shift_logits_orig.argmax(dim=-1)
        top1_merged = shift_logits_merged.argmax(dim=-1)
        top1_agree_sum += (top1_orig == top1_merged)[shift_mask].sum().item()

        total_tokens += shift_mask.sum().item()

        del logits_orig, logits_merged, logp_orig, logp_merged, p_orig
        torch.cuda.empty_cache()

    # Restore the original, unmodified experts before returning.
    install_experts(moe_block, [i, j], [orig_i, orig_j])

    mean_kl = kl_sum / total_tokens
    mean_ce_orig = ce_orig_sum / total_tokens
    mean_ce_merged = ce_merged_sum / total_tokens

    return {
        "Oracle_KL": mean_kl,
        "CrossEntropy_Delta": mean_ce_merged - mean_ce_orig,
        "Perplexity_Delta": math.exp(mean_ce_merged) - math.exp(mean_ce_orig),
        "Top1_Agreement": top1_agree_sum / total_tokens,
    }


## 12. Run the full sweep — existing metrics + oracle, for every expert pair

In [ ]:

all_pairs = list(itertools.combinations(range(num_experts), 2))
print(f"Total expert pairs at layer {LAYER_INDEX}: {len(all_pairs)}")

if MAX_PAIRS is not None:
    rng2 = random.Random(SEED)
    all_pairs = rng2.sample(all_pairs, min(MAX_PAIRS, len(all_pairs)))
    print(f"MAX_PAIRS set -> subsampled to {len(all_pairs)} pairs for this run")

rows = []
for (i, j) in tqdm(all_pairs, desc=f"Layer {LAYER_INDEX}: metrics + oracle per pair"):
    row = {
        "Layer": LAYER_INDEX,
        "Expert_A": i,
        "Expert_B": j,
        "Weight_Distance": weight_distance(i, j),
        "Weight_Cosine": weight_cosine(i, j),
        "Activation_Similarity": activation_similarity(i, j),
        "Output_Similarity": output_similarity(i, j),
        "Routing_Similarity": routing_similarity(i, j),
        # Usage_Frequency: schema calls for a single scalar per pair; we use the
        # mean of the two individual experts' usage rates. Per-expert rates are
        # available in `usage_frequency[]` above if finer-grained analysis is needed.
        "Usage_Frequency": float((usage_frequency[i] + usage_frequency[j]) / 2.0),
    }
    row.update(run_oracle_pair(
        model, moe_block, i, j, merge_operator,
        calib_input_ids, calib_attn_mask, batch_size=CALIB_BATCH_SIZE,
    ))
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(OUTPUT_DIR, "metrics.csv"), index=False)
print(f"Saved {len(results_df)} rows to {os.path.join(OUTPUT_DIR, 'metrics.csv')}")
results_df.head()


## 13. Oracle matrix — symmetric Expert x Expert KL divergence table

In [ ]:

oracle_matrix = np.full((num_experts, num_experts), np.nan)
for _, r in results_df.iterrows():
    a, b = int(r["Expert_A"]), int(r["Expert_B"])
    oracle_matrix[a, b] = r["Oracle_KL"]
    oracle_matrix[b, a] = r["Oracle_KL"]

oracle_matrix_df = pd.DataFrame(
    oracle_matrix,
    index=[f"E{k}" for k in range(num_experts)],
    columns=[f"E{k}" for k in range(num_experts)],
)
oracle_matrix_df.to_csv(os.path.join(OUTPUT_DIR, "oracle_matrix.csv"))
print(f"Saved {os.path.join(OUTPUT_DIR, 'oracle_matrix.csv')}")


## 14. Statistical analysis — Pearson & Spearman correlation vs Oracle_KL

In [ ]:

metric_cols = [
    "Weight_Distance", "Weight_Cosine", "Activation_Similarity",
    "Output_Similarity", "Routing_Similarity", "Usage_Frequency",
]

corr_rows = []
for col in metric_cols:
    pear_r, pear_p = pearsonr(results_df[col], results_df["Oracle_KL"])
    spear_r, spear_p = spearmanr(results_df[col], results_df["Oracle_KL"])
    corr_rows.append({
        "Metric": col,
        "Pearson_r": pear_r,
        "Pearson_p": pear_p,
        "Spearman_r": spear_r,
        "Spearman_p": spear_p,
        "_abs_pearson": abs(pear_r),
    })

corr_df = (
    pd.DataFrame(corr_rows)
    .sort_values("_abs_pearson", ascending=False)
    .drop(columns="_abs_pearson")
    .reset_index(drop=True)
)
corr_df.to_csv(os.path.join(OUTPUT_DIR, "correlations.csv"), index=False)
print(f"Saved {os.path.join(OUTPUT_DIR, 'correlations.csv')}")
print(corr_df.to_string(index=False))


## 15. Visualization

In [ ]:

# 1. Oracle heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(oracle_matrix_df, cmap="viridis", square=True)
plt.title(f"Oracle KL divergence — Layer {LAYER_INDEX} (expert x expert)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "oracle_heatmap.png"), dpi=150)
plt.show()


In [ ]:

# 2. Correlation matrix among all existing metrics + Oracle_KL
full_corr = results_df[metric_cols + ["Oracle_KL"]].corr(method="pearson")
plt.figure(figsize=(8, 6))
sns.heatmap(full_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Pearson correlation matrix — existing metrics vs Oracle_KL")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "correlation_matrix.png"), dpi=150)
plt.show()


In [ ]:

# 3. Scatter plot for every metric vs Oracle_KL
for col in metric_cols:
    r_val = corr_df.loc[corr_df["Metric"] == col, "Pearson_r"].values[0]
    plt.figure(figsize=(5, 4))
    plt.scatter(results_df[col], results_df["Oracle_KL"], alpha=0.5, s=12)
    plt.xlabel(col)
    plt.ylabel("Oracle_KL")
    plt.title(f"{col} vs Oracle_KL (Pearson r = {r_val:.3f})")
    plt.tight_layout()
    plt.savefig(os.path.join(SCATTER_DIR, f"{col}_vs_oracle_kl.png"), dpi=150)
    plt.show()


In [ ]:

# 4. Ranked merge list — safest merge candidates first (lowest Oracle_KL)
ranked = results_df.sort_values("Oracle_KL", ascending=True).reset_index(drop=True)
ranked.to_csv(os.path.join(OUTPUT_DIR, "ranked_merge_candidates.csv"), index=False)

top_n = min(20, len(ranked))
labels = [f"E{a}-E{b}" for a, b in zip(ranked["Expert_A"][:top_n], ranked["Expert_B"][:top_n])]

plt.figure(figsize=(8, max(4, top_n * 0.3)))
plt.barh(labels[::-1], ranked["Oracle_KL"][:top_n][::-1])
plt.xlabel("Oracle_KL (lower = safer merge)")
plt.title(f"Top {top_n} lowest-capability-loss merge candidates — Layer {LAYER_INDEX}")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ranked_merge_list.png"), dpi=150)
plt.show()

print("\nAll outputs saved under:", os.path.abspath(OUTPUT_DIR))



## Output files

| File | Contents |
|---|---|
| `metrics.csv` | One row per expert pair: all existing metrics + all oracle metrics |
| `oracle_matrix.csv` | Symmetric Expert x Expert `Oracle_KL` matrix |
| `correlations.csv` | Pearson/Spearman correlation of each metric vs `Oracle_KL`, ranked |
| `ranked_merge_candidates.csv` | All pairs sorted by `Oracle_KL` ascending |
| `oracle_heatmap.png` | Heatmap of the oracle matrix |
| `correlation_matrix.png` | Correlation matrix of all metrics + Oracle_KL |
| `ranked_merge_list.png` | Bar chart of the top-20 safest merge candidates |
| `scatterplots/*.png` | One scatter plot per metric vs `Oracle_KL` |

To evaluate a different layer, change `LAYER_INDEX` in the config cell and
re-run from Section 5 onward (sections 1–4 don't depend on layer choice).
